# Venue Homes -> Ward-Level Areal Interpolation

Pipeline:

1. Load `venues_homes.csv` and join venue coordinates from `venues-centroids.geojson` (on `id` <-> `venue_id`)
2. Decode `home_geohash6` to a centroid lat/lon
3. Build a grid GeoDataFrame: one polygon per venue's home geohash-6 cell, keeping all venue fields
4. Load `city-wards.geojson`
5. **Areal-weighted interpolation**: split each geohash-6 cell's `normalized_raw_stop_count`
   across whichever wards it overlaps, proportional to the overlap area, then sum per
   `(venue_id, ward)` -- this gives one row per venue per ward
6. Flag `home_ward = True` on the one row per venue where that venue's own point actually
   falls inside that ward's polygon

Requirements: `pandas`, `geopandas`, `shapely`, `pygeohash`
(`pip install pandas geopandas shapely pygeohash`)


## 0. Config

Adjust these to match your actual file schemas -- column names are guessed based on your
description; the notebook prints each file's columns as it loads so you can confirm/adjust.


In [1]:
import pandas as pd
import geopandas as gpd
import pygeohash as pgh
from shapely.geometry import box

# ---- file paths ----
VENUES_HOMES_CSV = "../../data/activity/venues_homes.csv"
VENUE_CENTROIDS_GEOJSON = "../../data/venues/tac-list/venues-centroids.geojson"
WARDS_GEOJSON = "../../data/geo/city-wards.geojson"

# ---- column names -- ADJUST if your files use different field names ----
VENUE_ID_COL = "venue_id"                        # id column in venues_homes.csv
CENTROID_ID_COL = "id"                           # id column in venues-centroids.geojson
VENUE_NAME_COL = "venue_name"                    # name column in venues-centroids.geojson (if present)
WARD_NAME_COL = "ward_name"                      # name column in city-wards.geojson
VALUE_COL = "normalized_raw_stop_count"          # value column in venues_homes.csv to interpolate
GEOHASH_COL = "home_geohash6"

RESULT_COL = f"{VALUE_COL}_by_ward"              # name of the interpolated output column


## 1. Load venues_homes.csv and join venue coordinates

In [2]:
venues_homes = pd.read_csv(VENUES_HOMES_CSV)
print("venues_homes columns:", venues_homes.columns.tolist())
venues_homes.head()


venues_homes columns: ['home_geohash6', 'venue_id', 'venue_name', 'time_period', 'raw_stop_count', 'visit_count', 'unique_devices', 'normalized_raw_stop_count', 'normalized_visit_count', 'normalized_unique_device_months']


,home_geohash6,venue_id,venue_name,time_period,raw_stop_count,visit_count,unique_devices,normalized_raw_stop_count,normalized_visit_count,normalized_unique_device_months
0,dpz2jc,1,InterAccess,all,2,2,2,1.324172,1.324172,1.324172
1,dpz2jf,1,InterAccess,all,2,2,2,1.531940,1.531940,1.531940
2,dpz2mt,1,InterAccess,all,1,1,1,0.583720,0.583720,0.583720
3,dpz2mx,1,InterAccess,all,1,1,1,0.714649,0.714649,0.714649
4,dpz2nh,1,InterAccess,all,4,4,2,2.364828,2.364828,1.151578


In [3]:
venue_centroids = gpd.read_file(VENUE_CENTROIDS_GEOJSON)
print("venue_centroids columns:", venue_centroids.columns.tolist())

# extract lat/lon from the point geometry
venue_centroids["venue_lon"] = venue_centroids.geometry.x
venue_centroids["venue_lat"] = venue_centroids.geometry.y

# make sure both join keys are the same dtype/format before merging
# (e.g. one side stored as int, the other as string -- this normalizes both to stripped strings)
venues_homes[VENUE_ID_COL] = venues_homes[VENUE_ID_COL].astype(str).str.strip()
venue_centroids[CENTROID_ID_COL] = venue_centroids[CENTROID_ID_COL].astype(str).str.strip()

join_cols = [CENTROID_ID_COL, "venue_lat", "venue_lon"]
if VENUE_NAME_COL in venue_centroids.columns:
    join_cols.append(VENUE_NAME_COL)
else:
    print(f"Note: '{VENUE_NAME_COL}' not found in venue_centroids -- skipping venue name join.")

venues_homes = venues_homes.merge(
    venue_centroids[join_cols],
    left_on=VENUE_ID_COL, right_on=CENTROID_ID_COL, how="left",
).drop(columns=[CENTROID_ID_COL])

n_missing = venues_homes["venue_lat"].isna().sum()
if n_missing:
    print(f"Warning: {n_missing} venues_homes rows did not match a centroid record on id.")

venues_homes.head()


venue_centroids columns: ['fid', 'venue_name', 'address', 'postal_code', 'max_attendee_capacity', 'total_facility_area_sqft', 'disciplines_supported', 'primary_discipline', 'ownership', 'operator', 'tac_funded_operator', 'tac_funded_resident', 'tac_funded_programming', 'year_opened', 'venue_description', 'owner', 'id', 'geometry']


,home_geohash6,venue_id,venue_name_x,time_period,raw_stop_count,visit_count,unique_devices,normalized_raw_stop_count,normalized_visit_count,normalized_unique_device_months,venue_lat,venue_lon,venue_name_y
0,dpz2jc,1,InterAccess,all,2,2,2,1.324172,1.324172,1.324172,43.641945,-79.423363,InterAccess
1,dpz2jf,1,InterAccess,all,2,2,2,1.531940,1.531940,1.531940,43.641945,-79.423363,InterAccess
2,dpz2mt,1,InterAccess,all,1,1,1,0.583720,0.583720,0.583720,43.641945,-79.423363,InterAccess
3,dpz2mx,1,InterAccess,all,1,1,1,0.714649,0.714649,0.714649,43.641945,-79.423363,InterAccess
4,dpz2nh,1,InterAccess,all,4,4,2,2.364828,2.364828,1.151578,43.641945,-79.423363,InterAccess


## 2. Decode home_geohash6 centroid coordinates

In [4]:
venues_homes[GEOHASH_COL] = venues_homes[GEOHASH_COL].astype(str).str.strip().str.lower()

def safe_decode(gh):
    try:
        lat, lon = pgh.decode(gh)
        return pd.Series({"home_geohash6_lat": lat, "home_geohash6_lon": lon})
    except Exception:
        return pd.Series({"home_geohash6_lat": None, "home_geohash6_lon": None})

venues_homes[["home_geohash6_lat", "home_geohash6_lon"]] = venues_homes[GEOHASH_COL].apply(safe_decode)
venues_homes.head()


,home_geohash6,venue_id,venue_name_x,time_period,raw_stop_count,visit_count,unique_devices,normalized_raw_stop_count,normalized_visit_count,normalized_unique_device_months,venue_lat,venue_lon,venue_name_y,home_geohash6_lat,home_geohash6_lon
0,dpz2jc,1,InterAccess,all,2,2,2,1.324172,1.324172,1.324172,43.641945,-79.423363,InterAccess,43.601990,-79.546509
1,dpz2jf,1,InterAccess,all,2,2,2,1.531940,1.531940,1.531940,43.641945,-79.423363,InterAccess,43.607483,-79.546509
2,dpz2mt,1,InterAccess,all,1,1,1,0.583720,0.583720,0.583720,43.641945,-79.423363,InterAccess,43.667908,-79.557495
3,dpz2mx,1,InterAccess,all,1,1,1,0.714649,0.714649,0.714649,43.641945,-79.423363,InterAccess,43.678894,-79.557495
4,dpz2nh,1,InterAccess,all,4,4,2,2.364828,2.364828,1.151578,43.641945,-79.423363,InterAccess,43.618469,-79.535522


## 3. Build the grid GeoDataFrame (one polygon per venue's home cell)

Unlike a deduplicated grid, this keeps **one row per venue** (with its full set of fields),
because the interpolation step below needs a per-venue value, not a per-cell aggregate. If
two venues share the same geohash-6 cell, they'll get identical (overlapping) polygons -- that's
expected here.


In [5]:
def geohash_to_polygon(gh):
    """True bounding-box polygon of a geohash string, using its exact error margins."""
    lat_c, lon_c, lat_err, lon_err = pgh.decode_exactly(gh)
    return box(lon_c - lon_err, lat_c - lat_err, lon_c + lon_err, lat_c + lat_err)

grid_gdf = venues_homes.copy()
grid_gdf["geometry"] = grid_gdf[GEOHASH_COL].apply(geohash_to_polygon)
grid_gdf = gpd.GeoDataFrame(grid_gdf, geometry="geometry", crs="EPSG:4326")

print(f"grid_gdf: {len(grid_gdf)} rows (one per venue)")
grid_gdf.head()


grid_gdf: 49257 rows (one per venue)


,home_geohash6,venue_id,venue_name_x,time_period,raw_stop_count,visit_count,unique_devices,normalized_raw_stop_count,normalized_visit_count,normalized_unique_device_months,venue_lat,venue_lon,venue_name_y,home_geohash6_lat,home_geohash6_lon,geometry
0,dpz2jc,1,InterAccess,all,2,2,2,1.324172,1.324172,1.324172,43.641945,-79.423363,InterAccess,43.601990,-79.546509,"POLYGON ((-79.54102 43.59924, -79.54102 43.604..."
1,dpz2jf,1,InterAccess,all,2,2,2,1.531940,1.531940,1.531940,43.641945,-79.423363,InterAccess,43.607483,-79.546509,"POLYGON ((-79.54102 43.60474, -79.54102 43.610..."
2,dpz2mt,1,InterAccess,all,1,1,1,0.583720,0.583720,0.583720,43.641945,-79.423363,InterAccess,43.667908,-79.557495,"POLYGON ((-79.552 43.66516, -79.552 43.67065, ..."
3,dpz2mx,1,InterAccess,all,1,1,1,0.714649,0.714649,0.714649,43.641945,-79.423363,InterAccess,43.678894,-79.557495,"POLYGON ((-79.552 43.67615, -79.552 43.68164, ..."
4,dpz2nh,1,InterAccess,all,4,4,2,2.364828,2.364828,1.151578,43.641945,-79.423363,InterAccess,43.618469,-79.535522,"POLYGON ((-79.53003 43.61572, -79.53003 43.621..."


## 4. Load ward polygons

Real-world ward/boundary files often carry a stray non-polygon row (a boundary-line artifact,
a label point) or an invalid self-intersecting polygon. Both will make `geopandas.overlay`
fail later with `NotImplementedError: ... contains mixed geometry types`, so this is cleaned
up once here, immediately after loading, so every downstream step (the cross-join, the
interpolation, the `home_ward` spatial join) sees a consistent, pure-polygon `wards_gdf`.


In [6]:
from shapely.ops import unary_union

def keep_polygons_only(gdf, label="layer"):
    """Repair invalid geometries and drop/collapse anything that isn't a polygon,
    so downstream steps (overlay, cross-joins, spatial joins) never see a mixed-type layer."""
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]

    types_before = gdf.geometry.geom_type.value_counts().to_dict()
    if len(types_before) > 1 or not set(types_before) <= {"Polygon", "MultiPolygon"}:
        print(f"{label}: geometry types found -> {types_before}")

    gdf["geometry"] = gdf.geometry.make_valid()

    def polys_only(geom):
        if geom is None or geom.is_empty:
            return None
        if geom.geom_type in ("Polygon", "MultiPolygon"):
            return geom
        if geom.geom_type == "GeometryCollection":
            polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
            if not polys:
                return None
            return polys[0] if len(polys) == 1 else unary_union(polys)
        return None  # drop stray points/lines entirely

    gdf["geometry"] = gdf["geometry"].apply(polys_only)
    n_dropped = int(gdf["geometry"].isna().sum())
    if n_dropped:
        print(f"{label}: dropped {n_dropped} non-polygonal row(s) after cleaning")

    return gdf[gdf["geometry"].notna()].reset_index(drop=True)


wards_gdf = gpd.read_file(WARDS_GEOJSON)
print("wards columns:", wards_gdf.columns.tolist())

if wards_gdf.crs is None:
    wards_gdf = wards_gdf.set_crs("EPSG:4326")
else:
    wards_gdf = wards_gdf.to_crs("EPSG:4326")

wards_gdf = keep_polygons_only(wards_gdf, "wards")

wards_gdf.head()


wards columns: ['ward_code', 'ward_name', 'geometry']
wards: geometry types found -> {'MultiPolygon': 25, 'MultiLineString': 21, 'LineString': 4}
wards: dropped 25 non-polygonal row(s) after cleaning


,ward_code,ward_name,geometry
0,07,Humber River-Black Creek,"MULTIPOLYGON (((-79.49105 43.7635, -79.49008 4..."
1,06,York Centre,"MULTIPOLYGON (((-79.44043 43.7634, -79.43998 4..."
2,18,Willowdale,"MULTIPOLYGON (((-79.39449 43.76157, -79.39461 ..."
3,11,University-Rosedale,"MULTIPOLYGON (((-79.39004 43.6905, -79.39004 4..."
4,19,Beaches-East York,"MULTIPOLYGON (((-79.29864 43.71515, -79.29837 ..."


## 5. Areal-weighted interpolation: geohash cell -> ward

For each venue's home cell, split `normalized_raw_stop_count` across the wards it overlaps,
in proportion to the fraction of the cell's area that falls in each ward. This is standard
areal weighting (assumes the value is spread uniformly within the source cell).

Steps:
1. Reproject to a local projected CRS (so area calculations are accurate in meters, not degrees)
2. Overlay the grid cells with the (already-cleaned) ward polygons to get the intersection pieces
3. `weight = piece_area / original_cell_area`
4. `interpolated_value = weight * normalized_raw_stop_count`
5. Sum by `(venue_id, ward)` -- a cell can rarely produce more than one piece in the same ward
   (e.g. an irregular ward boundary), so this collapses those back together


In [7]:
# reproject both layers to an accurate local CRS for area math
proj_crs = grid_gdf.estimate_utm_crs()
grid_proj = grid_gdf.to_crs(proj_crs)
wards_proj = wards_gdf.to_crs(proj_crs)

# slim the grid down before the overlay to avoid column name collisions with ward attributes,
# and run it through the same polygon-only cleaning as a safety net
grid_slim = grid_proj[[VENUE_ID_COL, VALUE_COL, "geometry"]].copy()
grid_slim["source_area"] = grid_slim.geometry.area
grid_slim = keep_polygons_only(grid_slim, "grid")

pieces = gpd.overlay(
    grid_slim,
    wards_proj[[WARD_NAME_COL, "geometry"]],
    how="intersection",
    keep_geom_type=True,
)

pieces["piece_area"] = pieces.geometry.area
pieces["area_weight"] = pieces["piece_area"] / pieces["source_area"]
pieces["interpolated_value"] = pieces["area_weight"] * pieces[VALUE_COL]

ward_venue_agg = (
    pieces.groupby([VENUE_ID_COL, WARD_NAME_COL], as_index=False)["interpolated_value"].sum()
    .rename(columns={"interpolated_value": RESULT_COL})
)

ward_venue_agg.head()


,venue_id,ward_name,normalized_raw_stop_count_by_ward
0,1,Beaches-East York,16.769873
1,1,Davenport,483.169165
2,1,Don Valley East,16.933730
3,1,Don Valley North,33.113227
4,1,Don Valley West,15.884746


## 6. Ensure every venue has a row for every ward (fill 0 where there's no overlap)


In [8]:
all_venues = grid_gdf[VENUE_ID_COL].unique()
all_wards = wards_gdf[WARD_NAME_COL].unique()

full_index = pd.MultiIndex.from_product([all_venues, all_wards], names=[VENUE_ID_COL, WARD_NAME_COL])
result = pd.DataFrame(index=full_index).reset_index()
result = result.merge(ward_venue_agg, on=[VENUE_ID_COL, WARD_NAME_COL], how="left")
result[RESULT_COL] = result[RESULT_COL].fillna(0.0)

print(f"rows: {len(result)}  (should be {len(all_venues)} venues x {len(all_wards)} wards = {len(all_venues) * len(all_wards)})")

# sanity check: per venue, the interpolated pieces should sum back to ~ the original value
# (a shortfall means part of that venue's home cell falls outside all ward polygons --
#  i.e. the wards layer doesn't fully cover that cell, which is a data-coverage question,
#  not a bug in the interpolation)
check = result.groupby(VENUE_ID_COL)[RESULT_COL].sum()
orig = grid_gdf.drop_duplicates(VENUE_ID_COL).set_index(VENUE_ID_COL)[VALUE_COL]
reconstruction_diff = (check - orig).abs()
print("Max reconstruction diff (0 = fully covered by wards):", reconstruction_diff.max())


rows: 2625  (should be 105 venues x 25 wards = 2625)
Max reconstruction diff (0 = fully covered by wards): 89657.49338911426


## 7. Attach venue attributes and select final columns

In [9]:
venue_attrs = venues_homes.drop_duplicates(VENUE_ID_COL).set_index(VENUE_ID_COL)

attr_cols = ["venue_lat", "venue_lon"]
if VENUE_NAME_COL in venue_attrs.columns:
    attr_cols.append(VENUE_NAME_COL)

result = result.merge(venue_attrs[attr_cols], left_on=VENUE_ID_COL, right_index=True, how="left")

final_cols = [VENUE_ID_COL]
if VENUE_NAME_COL in result.columns:
    final_cols.append(VENUE_NAME_COL)
final_cols += ["venue_lat", "venue_lon", WARD_NAME_COL, RESULT_COL]

result = result[final_cols]
result.head(20)


,venue_id,venue_lat,venue_lon,ward_name,normalized_raw_stop_count_by_ward
0,1,43.641945,-79.423363,Humber River-Black Creek,19.384094
1,1,43.641945,-79.423363,York Centre,19.312928
2,1,43.641945,-79.423363,Willowdale,9.942956
3,1,43.641945,-79.423363,University-Rosedale,106.780396
4,1,43.641945,-79.423363,Beaches-East York,16.769873
5,1,43.641945,-79.423363,Scarborough Southwest,13.030871
6,1,43.641945,-79.423363,Scarborough-Rouge Park,31.003561
7,1,43.641945,-79.423363,Scarborough North,4.041630
8,1,43.641945,-79.423363,Scarborough-Guildwood,14.423941
9,1,43.641945,-79.423363,Scarborough Centre,8.037240


## 8. `home_ward` flag

True on the one row per venue where that venue's own coordinate actually falls inside that
ward's polygon (a simple point-in-polygon spatial join).


In [10]:
venue_points_df = venue_attrs.reset_index()[[VENUE_ID_COL, "venue_lat", "venue_lon"]]
venue_points = gpd.GeoDataFrame(
    venue_points_df,
    geometry=gpd.points_from_xy(venue_points_df["venue_lon"], venue_points_df["venue_lat"]),
    crs="EPSG:4326",
)

venue_in_ward = gpd.sjoin(
    venue_points, wards_gdf[[WARD_NAME_COL, "geometry"]], how="left", predicate="within"
)[[VENUE_ID_COL, WARD_NAME_COL]].rename(columns={WARD_NAME_COL: "_actual_ward"})

result = result.merge(venue_in_ward, on=VENUE_ID_COL, how="left")
result["home_ward"] = result[WARD_NAME_COL] == result["_actual_ward"]
result = result.drop(columns=["_actual_ward"])

n_venues_unmatched = result.groupby(VENUE_ID_COL)["home_ward"].any().eq(False).sum()
if n_venues_unmatched:
    print(f"Note: {n_venues_unmatched} venues did not fall inside any ward polygon (home_ward all False).")

result.sort_values([VENUE_ID_COL, WARD_NAME_COL]).head(20)


,venue_id,venue_lat,venue_lon,ward_name,normalized_raw_stop_count_by_ward,home_ward
4,1,43.641945,-79.423363,Beaches-East York,16.769873,False
20,1,43.641945,-79.423363,Davenport,483.169165,True
19,1,43.641945,-79.423363,Don Valley East,16.933730,False
18,1,43.641945,-79.423363,Don Valley North,33.113227,False
11,1,43.641945,-79.423363,Don Valley West,15.884746,False
17,1,43.641945,-79.423363,Eglinton-Lawrence,24.693429,False
16,1,43.641945,-79.423363,Etobicoke Centre,10.522710,False
14,1,43.641945,-79.423363,Etobicoke North,18.483467,False
15,1,43.641945,-79.423363,Etobicoke-Lakeshore,21.165424,False
0,1,43.641945,-79.423363,Humber River-Black Creek,19.384094,False


## 8b. Attach venue_name directly from the raw CSV

Added as a final, standalone step: re-reads `venue_id -> venue_name` straight from
`venues_homes.csv` (rather than relying on `venue_name` surviving the earlier merges/joins
above -- e.g. if it got suffixed to `venue_name_x`/`venue_name_y` by a name collision with
another file upstream) and joins it onto `result` by `venue_id`. This is independent of
whatever happened to `venue_name` earlier in the notebook.


In [11]:
# pull a clean venue_id -> venue_name lookup directly from the source CSV
name_lookup = pd.read_csv(VENUES_HOMES_CSV)[[VENUE_ID_COL, VENUE_NAME_COL]].drop_duplicates(subset=VENUE_ID_COL)
name_lookup[VENUE_ID_COL] = name_lookup[VENUE_ID_COL].astype(str).str.strip()

# flag (don't silently hide) any venue_id that maps to more than one distinct name in the raw data
name_counts = pd.read_csv(VENUES_HOMES_CSV)[[VENUE_ID_COL, VENUE_NAME_COL]].drop_duplicates()
conflicts = name_counts.groupby(VENUE_ID_COL)[VENUE_NAME_COL].nunique()
conflicts = conflicts[conflicts > 1]
if len(conflicts):
    print(f"Warning: {len(conflicts)} venue_id(s) have more than one distinct venue_name in the raw CSV: {conflicts.index.tolist()}")

# make sure the join key matches dtype/format on both sides
result[VENUE_ID_COL] = result[VENUE_ID_COL].astype(str).str.strip()

# drop any stale venue_name-ish column left over from earlier steps, then join the clean one in
result = result.drop(columns=[c for c in result.columns if c.startswith(VENUE_NAME_COL)], errors="ignore")
result = result.merge(name_lookup, on=VENUE_ID_COL, how="left")

n_missing_name = result[VENUE_NAME_COL].isna().sum()
if n_missing_name:
    print(f"Warning: {n_missing_name} result rows did not get a venue_name (no matching venue_id in the CSV).")

# put venue_name right after venue_id for readability
cols = result.columns.tolist()
cols.remove(VENUE_NAME_COL)
cols.insert(cols.index(VENUE_ID_COL) + 1, VENUE_NAME_COL)
result = result[cols]

result.head()


,venue_id,venue_name,venue_lat,venue_lon,ward_name,normalized_raw_stop_count_by_ward,home_ward
0,1,InterAccess,43.641945,-79.423363,Humber River-Black Creek,19.384094,False
1,1,InterAccess,43.641945,-79.423363,York Centre,19.312928,False
2,1,InterAccess,43.641945,-79.423363,Willowdale,9.942956,False
3,1,InterAccess,43.641945,-79.423363,University-Rosedale,106.780396,False
4,1,InterAccess,43.641945,-79.423363,Beaches-East York,16.769873,False


## 9. Result

`result` has one row per `(venue_id, ward)` combination, with:

- `venue_id`, `venue_name`, `venue_lat`, `venue_lon` -- the venue
- `ward_name` -- the ward
- the interpolated value column (`normalized_raw_stop_count_by_ward` by default) -- estimated
  count of stops from that ward that ended up at that venue, area-weighted from the home
  geohash-6 cell
- `home_ward` -- True only on the row where the venue itself is physically located

In [12]:
result.to_csv("../../data/activity/ward_to_venue_activity.csv", index=False)